# Uncertainty Quantification for Scientific Machine Learning using Sparse Variational Gaussian Process Kolmogorov-Arnold Networks (SVGP-KAN)

**Paper:** Ju, Y.S. (2026). *Uncertainty Quantification for Scientific Machine Learning using Sparse Variational Gaussian Process Kolmogorov-Arnold Networks (SVGP KAN).* Machine Learning: Science and Technology, in press (Accepted Manuscript). https://doi.org/10.1088/2632-2153/ae5502

**Carpeta origen:** `Kolmogorov-Arnold Networks/Papers/Ciencia, energía nuclear y química/Uncertainty quantification for scientific machine learning using sparse variational Gaussian process Kolmogorov-Arnold networks (SVGP KAN).pdf`

## Como se usan las KAN en este paper

Una red Kolmogorov-Arnold (KAN) transforma un vector de entrada mediante capas aditivas de funciones univariadas colocadas en las aristas (edges) en vez de activaciones fijas en los nodos:

$$y_j = \sum_{i=1}^{D_{in}} \phi_{j,i}(x_i) + b_j$$

En las implementaciones deterministas cada $\phi_{j,i}$ se parametriza con splines-B, polinomios de Chebyshev, etc. Este paper sustituye esa parametrizacion fija por un **proceso Gaussiano (GP) de media cero** sobre cada arista:

$$\phi_{j,i} \sim \mathcal{GP}(0, k(\cdot,\cdot;\theta_{j,i})), \qquad k_{SE}(x,x';\sigma_f^2,\ell) = \sigma_f^2\exp\left(-\frac{(x-x')^2}{2\ell^2}\right)$$

con nucleo RBF (squared-exponential). Como la inferencia GP exacta cuesta $O(N^3)$, el paper aplica **inferencia variacional dispersa** (Titsias 2009; Hensman et al. 2013): cada arista tiene $M \ll N$ puntos de induccion $\mathbf{Z}_{j,i}$ con variables de induccion $\mathbf{u}_{j,i}=\phi_{j,i}(\mathbf{Z}_{j,i})$, y una posterior variacional diagonal $q(\mathbf{u}_{j,i})=\mathcal{N}(\mathbf{m}_{j,i},\mathbf{S}_{j,i})$. La media y varianza predictivas en un punto $x$ son:

$$\mu_{j,i}(x)=\mathbf{k}_x^\top\mathbf{K}_{ZZ}^{-1}\mathbf{m}_{j,i}, \qquad \sigma_{j,i}^2(x)=\underbrace{\mathbf{k}_x^\top\mathbf{K}_{ZZ}^{-1}\mathbf{S}_{j,i}\mathbf{K}_{ZZ}^{-1}\mathbf{k}_x}_{V_{proj}} + \underbrace{\left(k(x,x)-\mathbf{k}_x^\top\mathbf{K}_{ZZ}^{-1}\mathbf{k}_x\right)}_{V_{orth}}$$

$V_{proj}$ (varianza proyectada) captura la incertidumbre transmitida por los puntos de induccion; $V_{orth}$ (varianza ortogonal, error de Nystrom) mide la informacion perdida por la representacion dispersa y **garantiza que la varianza predictiva vuelva al prior $\sigma_f^2$** cuando $x$ se aleja de los puntos de induccion, el mecanismo geometrico que el paper usa para deteccion de datos fuera de distribucion (OOD). El entrenamiento maximiza el **ELBO**:

$$\mathcal{L}_{ELBO}=\mathbb{E}_{q(\phi)}[\log p(\mathbf{y}\mid\phi)] - \lambda\cdot \mathrm{KL}[q(\mathbf{u})\,\|\,p(\mathbf{u})], \qquad \mathrm{KL}[q\|p]=\frac{1}{2}\left(\mathrm{tr}(\mathbf{K}_{ZZ}^{-1}\mathbf{S})+\mathbf{m}^\top\mathbf{K}_{ZZ}^{-1}\mathbf{m}-M+\log\frac{|\mathbf{K}_{ZZ}|}{|\mathbf{S}|}\right)$$

Para propagar incertidumbre a traves de capas profundas (la salida Gaussiana de una capa es la entrada incierta de la siguiente), el paper usa **emparejamiento analitico de momentos** (moment matching, Girard et al. 2002) mediante los estadisticos $\psi$: si $x\sim\mathcal{N}(\mu_x,\sigma_x^2)$,

$$[\psi_1]_m = \mathbb{E}_{p(x)}[k(x,z_m)] = \sigma_f^2\left(\frac{\ell^2}{\ell^2+\sigma_x^2}\right)^{1/2}\exp\left(-\frac{(z_m-\mu_x)^2}{2(\ell^2+\sigma_x^2)}\right)$$

y de forma analoga $\psi_2$ (estadistico de segundo orden, $[\psi_2]_{m,m'}=\mathbb{E}_{p(x)}[k(x,z_m)k(x,z_{m'})]$) permite obtener $\mathbb{E}[\mu_\phi(x)^2]$ y por tanto la varianza epistemica adicional que introduce la incertidumbre de entrada: $\mathbb{V}[y]=\sum_i\left(\mathbb{E}_{p(x)}[\sigma_\phi^2(x_i)]+\mathbb{V}_{p(x)}[\mu_\phi(x_i)]\right)$. El paper da la formula de $\psi_1$ explicitamente y remite a Girard et al. para la teoria general de $\psi_2$; en este cuaderno derivamos $\psi_2$ en forma cerrada para el nucleo RBF y la implementamos explicitamente.

Finalmente, el ruido de observacion heteroscedastico se modela con una **segunda GP dispersa sobre el log-varianza**: $\log\sigma^2_{noise}(x)\sim\mathcal{GP}(m_{noise}(x),k_{noise}(x,x'))$, de forma que el modelo separa incertidumbre aleatoria (ruido de medicion) de incertidumbre epistemica (falta de datos).

El paper valida el marco con tres estudios (campo escalar 2D con ruido heteroscedastico, prediccion multi-paso de adveccion-difusion, y deteccion OOD con autoencoders convolucionales). Este cuaderno reproduce fielmente el **mecanismo central** &mdash; capa KAN-GP dispersa, ELBO variacional, descomposicion $V_{proj}/V_{orth}$, y propagacion analitica de momentos mediante $\psi_1,\psi_2$ &mdash; sobre un **problema de regresion 1D sintetico con ruido heteroscedastico dependiente de la amplitud de la funcion** (analogo simplificado del Estudio A del paper), e incluye una demostracion de la reversion de la varianza al prior fuera del dominio de entrenamiento (analoga al mecanismo de deteccion OOD del Estudio C).

## Repositorio publico

El paper declara su repositorio oficial en la seccion "Data Availability": "The SVGP-KAN code library is available in https://github.com/sungjuGit/svgp-kan."

- **sungjuGit/svgp-kan** &mdash; https://github.com/sungjuGit/svgp-kan (modulo `svgp_kan/` con las clases `GPKANLayer`, `GPKAN`, `GPKANRegressor`, `SVGPUNet`)

In [ ]:
# Instalacion de dependencias (ejecutar si no estan ya instaladas en el entorno)
%pip install -q torch numpy matplotlib

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)
device = torch.device('cpu')  # el modelo es pequeno; CPU es suficiente y evita problemas de portabilidad
print('Device:', device)

## 1. Nucleo RBF y arista KAN-GP dispersa (Secciones 2.1-2.4 del paper)

Implementamos la clase `SVGPKANEdge`, que representa una unica arista $\phi_{j,i}$ del paper: un proceso Gaussiano disperso con nucleo RBF, $M$ puntos de induccion, y posterior variacional diagonal $q(\mathbf{u})=\mathcal{N}(\mathbf{m},\mathbf{S})$.

Cada arista expone:
- `kl()`: la divergencia KL cerrada de la Seccion 2.2.
- `psi1(mu_x, var_x)`: el estadistico $\psi_1$ dado explicitamente en el paper (Seccion 2.4), que se reduce al vector de kernel exacto $\mathbf{k}_x$ cuando la entrada es determinista ($\sigma_x^2=0$).
- `psi2(mu_x, var_x)`: el estadistico de segundo orden $[\psi_2]_{m,m'}=\mathbb{E}_{p(x)}[k(x,z_m)k(x,z_{m'})]$. El paper remite a Girard et al. (2002) para la teoria general sin dar la formula cerrada; aqui la derivamos para el nucleo RBF completando cuadrados en la integral Gaussiana:

$$[\psi_2]_{m,m'} = \sigma_f^4\sqrt{\frac{\ell^2}{\ell^2+2\sigma_x^2}}\exp\left(-\frac{(z_m-z_{m'})^2}{4\ell^2}\right)\exp\left(-\frac{(\mu_x-\tfrac{z_m+z_{m'}}{2})^2}{\ell^2+2\sigma_x^2}\right)$$

- `predict(mu_x, var_x)`: devuelve la media y varianza predictivas totales. Para entrada determinista usa la formula exacta $V_{proj}=\mathbf{k}_x^\top\mathbf{K}_{ZZ}^{-1}\mathbf{S}\mathbf{K}_{ZZ}^{-1}\mathbf{k}_x$ y la aproximacion eficiente que el propio paper da para $V_{orth}$ (Seccion 2.3): $V_{orth}\approx\sigma_f^2(1-\sum_m\psi_{1,m}^2/\sigma_f^4)$. Para entrada incierta (necesaria al encadenar capas, Seccion 2.4) usa $\psi_2$ para obtener $\mathbb{E}_{p(x)}[\sigma_\phi^2(x)]$ y $\mathbb{V}_{p(x)}[\mu_\phi(x)]=\mathbb{E}[\mu_\phi(x)^2]-\mathbb{E}[\mu_\phi(x)]^2$, con $\mathbb{E}[\mu_\phi(x)^2]=\boldsymbol{\alpha}^\top\Psi_2\boldsymbol{\alpha}$, $\boldsymbol{\alpha}=\mathbf{K}_{ZZ}^{-1}\mathbf{m}$.

La clase `SVGPKANLayer` agrupa $D_{in}\times D_{out}$ aristas y realiza la agregacion aditiva $y_j=\sum_i\phi_{j,i}(x_i)+b_j$ con independencia mean-field entre aristas (Seccion 2.4), y calcula la KL de la capa como el **promedio** sobre las aristas (pagina 12 del paper: esto evita tener que reajustar $\lambda$ al cambiar el ancho de la capa).

In [ ]:
JITTER = 1e-5


def rbf_kernel(x1, x2, sigma_f2, ell2):
    """Nucleo RBF (squared-exponential), Seccion 2.1, Eq. k_SE."""
    diff2 = (x1.unsqueeze(-1) - x2.unsqueeze(-2)) ** 2
    return sigma_f2 * torch.exp(-0.5 * diff2 / ell2)


class SVGPKANEdge(nn.Module):
    """Una arista KAN phi_{j,i}(x) modelada como GP disperso (Secciones 2.1, 2.2, 2.3, 2.4)."""

    def __init__(self, n_inducing=12, x_range=(-3.0, 3.0), init_ell=1.0, init_sigma_f=1.0):
        super().__init__()
        z_init = torch.linspace(x_range[0], x_range[1], n_inducing)
        self.Z = nn.Parameter(z_init.clone())                          # puntos de induccion Z_{j,i}
        # ruido pequeno para romper la simetria entre aristas de una misma capa: sin esto, todas las
        # aristas D_in x D_out de una capa parten de parametros identicos y reciben el mismo gradiente,
        # por lo que permanecerian identicas durante todo el entrenamiento (colapso a ancho efectivo 1).
        self.q_mu = nn.Parameter(0.1 * torch.randn(n_inducing))         # media variacional m_{j,i}
        self.q_log_var = nn.Parameter(torch.full((n_inducing,), -2.0))  # log-diagonal de S_{j,i}
        self.log_ell = nn.Parameter(torch.log(torch.tensor(float(init_ell))))
        self.log_sigma_f = nn.Parameter(torch.log(torch.tensor(float(init_sigma_f))))

    @property
    def ell2(self):
        return torch.exp(self.log_ell) ** 2

    @property
    def sigma_f2(self):
        return torch.exp(self.log_sigma_f) ** 2

    @property
    def q_var(self):
        return torch.exp(self.q_log_var)  # diagonal de S (> 0 por construccion)

    def Kzz(self):
        M = self.Z.shape[0]
        K = rbf_kernel(self.Z, self.Z, self.sigma_f2, self.ell2)
        return K + JITTER * torch.eye(M, device=self.Z.device)

    def psi1(self, mu_x, var_x):
        """[psi_1]_m = E_{x~N(mu_x,var_x)}[k(x,z_m)] (formula explicita del paper, Sec. 2.4)."""
        denom = self.ell2 + var_x.unsqueeze(-1)                       # (N,1)
        diff2 = (self.Z.unsqueeze(0) - mu_x.unsqueeze(-1)) ** 2        # (N,M)
        scale = self.sigma_f2 * torch.sqrt(self.ell2 / denom)
        return scale * torch.exp(-0.5 * diff2 / denom)                 # (N,M)

    def psi2(self, mu_x, var_x):
        """[psi_2]_{m,m'} = E_x[k(x,z_m) k(x,z_m')], forma cerrada RBF (derivada de Girard et al. 2002)."""
        Zm = self.Z.unsqueeze(0).unsqueeze(-1)      # (1,M,1)
        Zm2 = self.Z.unsqueeze(0).unsqueeze(-2)     # (1,1,M)
        c = 0.5 * (Zm + Zm2)                         # punto medio (z_m+z_m')/2 -> (1,M,M)
        d2 = (Zm - Zm2) ** 2                          # (z_m - z_m')^2 -> (1,M,M)
        denom = self.ell2 + 2.0 * var_x.view(-1, 1, 1)    # (N,1,1)
        mu = mu_x.view(-1, 1, 1)
        scale = self.sigma_f2 ** 2 * torch.sqrt(self.ell2 / denom)
        expo = -d2 / (4.0 * self.ell2) - (mu - c) ** 2 / denom
        return scale * torch.exp(expo)                # (N,M,M)

    def kl(self):
        """KL[q(u)||p(u)] cerrada, Seccion 2.2 del paper."""
        Kzz = self.Kzz()
        L = torch.linalg.cholesky(Kzz)
        Kzz_inv_m = torch.cholesky_solve(self.q_mu.unsqueeze(-1), L).squeeze(-1)
        Kzz_inv = torch.cholesky_solve(torch.eye(Kzz.shape[0], device=Kzz.device), L)
        trace_term = torch.sum(torch.diagonal(Kzz_inv) * self.q_var)
        quad_term = self.q_mu @ Kzz_inv_m
        logdet_Kzz = 2.0 * torch.sum(torch.log(torch.diagonal(L)))
        logdet_S = torch.sum(self.q_log_var)
        M = self.Z.shape[0]
        return 0.5 * (trace_term + quad_term - M + logdet_Kzz - logdet_S)

    def predict(self, mu_x, var_x=None, return_components=False):
        """Media y varianza predictivas totales.

        Si var_x es todo cero (entrada determinista) usa las formulas exactas de la Sec. 2.2-2.3.
        Si var_x > 0 (entrada incierta propagada desde una capa anterior) usa emparejamiento
        analitico de momentos con psi_1, psi_2 (Sec. 2.4).
        """
        if var_x is None:
            var_x = torch.zeros_like(mu_x)
        Kzz = self.Kzz()
        L = torch.linalg.cholesky(Kzz)

        psi1 = self.psi1(mu_x, var_x)                                       # (N,M)
        Kzz_inv_m = torch.cholesky_solve(self.q_mu.unsqueeze(-1), L).squeeze(-1)  # (M,)
        mean = psi1 @ Kzz_inv_m                                             # (N,)

        deterministic = torch.all(var_x == 0)

        if deterministic:
            # V_proj exacto: k_x^T Kzz^-1 S Kzz^-1 k_x  (aqui psi1 == k_x)
            Kzz_inv_psi1 = torch.cholesky_solve(psi1.T, L).T                # (N,M) = Kzz^-1 k_x por fila
            v_proj = (Kzz_inv_psi1 ** 2 * self.q_var.unsqueeze(0)).sum(-1)
            # V_orth: aproximacion eficiente dada explicitamente por el paper (Sec. 2.3)
            v_orth_ratio = torch.clamp((psi1 ** 2).sum(-1) / (self.sigma_f2 ** 2), max=1.0)
            v_orth = self.sigma_f2 * (1.0 - v_orth_ratio)
            total_var = v_proj + v_orth
            if return_components:
                return mean, v_proj, v_orth
            return mean, total_var
        else:
            # Emparejamiento de momentos completo via psi_2 (Sec. 2.4)
            psi2 = self.psi2(mu_x, var_x)                                   # (N,M,M)
            Kzz_inv = torch.cholesky_solve(torch.eye(Kzz.shape[0], device=Kzz.device), L)
            A = Kzz_inv @ torch.diag(self.q_var) @ Kzz_inv                  # (M,M)
            e_v_proj = torch.einsum('ij,nij->n', A, psi2)
            diag_psi2 = torch.diagonal(psi2, dim1=-2, dim2=-1).sum(-1)
            e_v_orth = self.sigma_f2 * (1.0 - torch.clamp(diag_psi2 / self.sigma_f2 ** 2, max=1.0))
            e_sigma2 = torch.clamp(e_v_proj, min=0.0) + e_v_orth            # E_p(x)[sigma_phi^2(x)]

            e_mu2 = torch.einsum('m,nmk,k->n', Kzz_inv_m, psi2, Kzz_inv_m)  # E_p(x)[mu_phi(x)^2]
            var_mu = torch.clamp(e_mu2 - mean ** 2, min=0.0)                # V_p(x)[mu_phi(x)]

            total_var = e_sigma2 + var_mu
            if return_components:
                return mean, e_sigma2, var_mu
            return mean, total_var

In [ ]:
class SVGPKANLayer(nn.Module):
    """Capa KAN con aristas GP dispersas: y_j = sum_i phi_{j,i}(x_i) + b_j (Sec. 2.1, Fig. 1a)."""

    def __init__(self, d_in, d_out, n_inducing=12, x_range=(-3.0, 3.0), init_ell=1.0):
        super().__init__()
        self.d_in, self.d_out = d_in, d_out
        self.edges = nn.ModuleList([
            SVGPKANEdge(n_inducing=n_inducing, x_range=x_range, init_ell=init_ell)
            for _ in range(d_in * d_out)
        ])
        self.bias = nn.Parameter(torch.zeros(d_out))

    def edge(self, j, i):
        return self.edges[j * self.d_in + i]

    def forward(self, mu_x, var_x=None):
        """mu_x, var_x: (N, d_in). Devuelve mu_y, var_y: (N, d_out)."""
        N = mu_x.shape[0]
        if var_x is None:
            var_x = torch.zeros_like(mu_x)
        mu_cols, var_cols = [], []
        for j in range(self.d_out):
            m_acc = torch.zeros(N, device=mu_x.device)
            v_acc = torch.zeros(N, device=mu_x.device)
            for i in range(self.d_in):
                e = self.edge(j, i)
                m, v = e.predict(mu_x[:, i], var_x[:, i])
                m_acc = m_acc + m
                v_acc = v_acc + v            # suma de varianzas: independencia mean-field entre aristas
            mu_cols.append(m_acc + self.bias[j])
            var_cols.append(v_acc)
        return torch.stack(mu_cols, dim=1), torch.stack(var_cols, dim=1)

    def kl(self):
        """KL promedio por arista (pagina 12: independiente del ancho D_in x D_out de la capa)."""
        kls = torch.stack([e.kl() for e in self.edges])
        return kls.mean()

## 2. Modelo completo: red SVGP-KAN de dos capas + GP de ruido heteroscedastico (Secciones 2.4, 2.5)

El regresor combina tres `SVGPKANLayer`:

1. **Capa 1** ($1\to H$): recibe $x$ determinista y produce $H$ salidas ocultas inciertas $(\mu_h,\sigma_h^2)$, cada una un GP disperso independiente.
2. **Capa 2** ($H\to 1$): recibe las $H$ salidas inciertas de la capa 1 como **entrada incierta** y usa el emparejamiento analitico de momentos ($\psi_1,\psi_2$) para producir la media y varianza predictivas de la senal $f(x)$. Esta es la varianza **epistemica** (Seccion 2.4), que decae en zonas con datos y revierte al prior lejos de ellos.
3. **Capa de ruido** ($1\to 1$): una GP dispersa separada sobre $x$ que modela $\log\sigma^2_{noise}(x)$ (Seccion 2.5), dando la varianza **aleatoria** (heteroscedastica).

La perdida de entrenamiento es el negativo del ELBO (Seccion 2.2, Eq. $\mathcal{L}_{ELBO}$), con la log-verosimilitud esperada aproximada por una Gaussiana con varianza total $\sigma^2_{epist}+\sigma^2_{noise}$ (Seccion 2.5, Eq. de $p(y\mid \mathbf{x},f,\sigma^2_{noise})$) y la KL sumada sobre las tres capas, ponderada por $\lambda$:

$$\mathrm{NLL}=\sum_{n=1}^N\left[\tfrac12\log(2\pi\sigma_n^2) + \tfrac{(y_n-\mu_n)^2}{2\sigma_n^2}\right], \qquad \mathcal{L}=\mathrm{NLL} + \lambda\sum_{\text{capas}}\mathrm{KL}_{\text{capa}}$$

Usamos $\lambda=0.01$, el valor tipico reportado por el paper para sus experimentos (Seccion 2.2, Tabla 3 del paper usa este mismo orden de magnitud).

In [ ]:
class SVGPKANRegressor(nn.Module):
    """Red SVGP-KAN 1 -> H -> 1 (senal) mas una GP dispersa separada 1 -> 1 (ruido, Sec. 2.5)."""

    def __init__(self, n_hidden=6, n_inducing=12, x_range=(-3.0, 3.0)):
        super().__init__()
        span = x_range[1] - x_range[0]
        self.layer1 = SVGPKANLayer(1, n_hidden, n_inducing=n_inducing, x_range=x_range, init_ell=span / 6)
        self.layer2 = SVGPKANLayer(n_hidden, 1, n_inducing=n_inducing, x_range=(-2.0, 2.0), init_ell=0.8)
        self.noise_layer = SVGPKANLayer(1, 1, n_inducing=n_inducing, x_range=x_range, init_ell=span / 6)

    def forward(self, x):
        x = x.view(-1, 1)
        mu_h, var_h = self.layer1(x)                            # Sec. 2.1-2.3: capa oculta incierta
        var_h = var_h.clamp(max=50.0)                            # estabilidad numerica durante el entrenamiento
        mu_y, var_epistemic = self.layer2(mu_h, var_h)           # Sec. 2.4: emparejamiento de momentos
        log_var_noise, _ = self.noise_layer(x)                   # Sec. 2.5: GP secundaria del ruido
        var_noise = torch.exp(log_var_noise.clamp(-6.0, 3.0))
        return mu_y.squeeze(-1), var_epistemic.squeeze(-1), var_noise.squeeze(-1)

    def kl(self):
        return self.layer1.kl() + self.layer2.kl() + self.noise_layer.kl()


def gaussian_nll(y, mu, var):
    var = var.clamp(min=1e-6)
    return 0.5 * torch.log(2 * np.pi * var) + 0.5 * (y - mu) ** 2 / var


def elbo_loss(model, x, y, lam=0.01):
    """Perdida = -ELBO = NLL_gaussiana(y | mu, var_epistemica+var_ruido) + lambda * KL total."""
    mu_y, var_epi, var_noise = model(x)
    var_total = var_epi + var_noise
    nll = gaussian_nll(y, mu_y, var_total).sum()
    kl = model.kl()
    loss = nll + lam * kl
    return loss, nll, kl, var_epi, var_noise

## 3. Problema sintetico 1D con ruido heteroscedastico (analogo al Estudio A del paper)

El Estudio A del paper (Seccion 3.2) usa un campo escalar $\omega(x,y)=\sin(x)\cos(y)$ con ruido de observacion que escala con la amplitud local del campo: $\sigma_{obs}(\omega)=0.01+0.15|\omega|$. Reproducimos la misma idea en 1D para poder visualizar las bandas de incertidumbre con claridad:

$$f(x)=\sin(2x)+0.5\cos(5x), \qquad \sigma_{noise}(x) = 0.05 + 0.2\,|f(x)|, \qquad y = f(x) + \sigma_{noise}(x)\cdot\eta,\ \ \eta\sim\mathcal{N}(0,1)$$

Igual que en el paper, el ruido es mas fuerte donde la funcion tiene mayor amplitud y casi nulo cerca de sus cruces por cero, creando una estructura heteroscedastica que el modelo debe aprender sin conocer la funcion generadora de ruido.

In [ ]:
def true_function(x):
    return np.sin(2 * x) + 0.5 * np.cos(5 * x)


def noise_std(f):
    return 0.05 + 0.2 * np.abs(f)


X_RANGE = (-3.0, 3.0)
N_TRAIN, N_TEST = 300, 100

rng = np.random.default_rng(0)
x_train_np = rng.uniform(*X_RANGE, N_TRAIN)
f_train = true_function(x_train_np)
sigma_train = noise_std(f_train)
y_train_np = f_train + rng.normal(0, 1, N_TRAIN) * sigma_train

x_test_np = rng.uniform(*X_RANGE, N_TEST)
f_test = true_function(x_test_np)
sigma_test = noise_std(f_test)
y_test_np = f_test + rng.normal(0, 1, N_TEST) * sigma_test

x_train = torch.tensor(x_train_np, dtype=torch.float32, device=device)
y_train = torch.tensor(y_train_np, dtype=torch.float32, device=device)
x_test = torch.tensor(x_test_np, dtype=torch.float32, device=device)
y_test = torch.tensor(y_test_np, dtype=torch.float32, device=device)

model = SVGPKANRegressor(n_hidden=6, n_inducing=12, x_range=X_RANGE).to(device)

with torch.no_grad():
    mu0, var_epi0, var_noise0 = model(x_train[:5])
print('Prueba de forward pass inicial (5 muestras):')
print('mu:', mu0.numpy())
print('var epistemica:', var_epi0.numpy())
print('var ruido:', var_noise0.numpy())

## 4. Entrenamiento variacional (maximizacion del ELBO)

Entrenamos con Adam minimizando $-\mathcal{L}_{ELBO}$ sobre todo el conjunto de entrenamiento en cada epoca (sin mini-batching, dado el tamano reducido del problema). Registramos por separado el termino de verosimilitud (NLL) y el termino de regularizacion (KL) para verificar que el ELBO evoluciona de forma razonable: la NLL debe decrecer mientras la KL crece moderadamente (el posterior variacional se aleja del prior para explicar los datos), sin que aparezcan valores NaN o divergencias.

In [ ]:
LAMBDA_KL = 0.01
N_EPOCHS = 800
LR = 5e-3

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
history = {'loss': [], 'nll': [], 'kl': []}

for epoch in range(N_EPOCHS):
    optimizer.zero_grad()
    loss, nll, kl, _, _ = elbo_loss(model, x_train, y_train, lam=LAMBDA_KL)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=10.0)
    optimizer.step()

    history['loss'].append(loss.item())
    history['nll'].append(nll.item())
    history['kl'].append(kl.item())

    if epoch % 100 == 0 or epoch == N_EPOCHS - 1:
        print(f'epoca {epoch:4d} | ELBO_loss={loss.item():10.3f} | NLL={nll.item():10.3f} | KL_total={kl.item():8.4f}')

assert not np.isnan(history['loss']).any(), 'La perdida contiene NaN'
print('\nEntrenamiento completado sin NaN.')

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot(history['loss'], color='tab:blue')
axes[0].set_title('Perdida total (-ELBO)'); axes[0].set_xlabel('epoca'); axes[0].set_ylabel('loss')
axes[1].plot(history['nll'], label='NLL (verosimilitud)', color='tab:orange')
axes[1].plot(history['kl'], label='KL total (suma de 3 capas)', color='tab:green')
axes[1].set_title('Componentes del ELBO'); axes[1].set_xlabel('epoca'); axes[1].legend()
plt.tight_layout(); plt.show()

## 5. Resultados: calibracion de la incertidumbre aleatoria (analogo a la Tabla 3, Estudio A)

Evaluamos en una malla densa y en el conjunto de test independiente. Comparamos la media y las bandas de confianza $\pm1\sigma,\pm2\sigma,\pm3\sigma$ (con $\sigma^2=\sigma^2_{epistemica}+\sigma^2_{ruido}$) contra la funcion verdadera y el ruido conocido, replicando las metricas de calibracion del paper (Seccion 4.1, Tabla 3): NLL, RMSE, correlacion de Pearson $\rho$ entre la desviacion estandar predicha y el error absoluto real, y cobertura empirica a $2\sigma$.

In [ ]:
model.eval()
x_grid_np = np.linspace(*X_RANGE, 400)
x_grid = torch.tensor(x_grid_np, dtype=torch.float32, device=device)

with torch.no_grad():
    mu_grid, var_epi_grid, var_noise_grid = model(x_grid)
    var_total_grid = var_epi_grid + var_noise_grid
    std_total_grid = torch.sqrt(var_total_grid)

    mu_test, var_epi_test, var_noise_test = model(x_test)
    var_total_test = var_epi_test + var_noise_test
    std_total_test = torch.sqrt(var_total_test)

mu_grid_np = mu_grid.numpy(); std_grid_np = std_total_grid.numpy()
f_grid_np = true_function(x_grid_np)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(x_grid_np, f_grid_np, 'k--', lw=1.5, label='f(x) verdadera')
ax.plot(x_grid_np, mu_grid_np, color='tab:blue', lw=2, label='media predicha SVGP-KAN')
for k, alpha in zip([1, 2, 3], [0.35, 0.22, 0.12]):
    ax.fill_between(x_grid_np, mu_grid_np - k * std_grid_np, mu_grid_np + k * std_grid_np,
                     color='tab:blue', alpha=alpha, label=f'±{k}σ' if k in (1, 2) else None)
ax.scatter(x_train_np, y_train_np, s=8, color='gray', alpha=0.5, label='datos de entrenamiento')
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.legend(loc='upper right', fontsize=8)
ax.set_title('Regresion SVGP-KAN con incertidumbre heteroscedastica')
plt.tight_layout(); plt.show()

# --- Metricas de calibracion (analogas a la Tabla 3 del paper) ---
mu_test_np = mu_test.numpy(); std_test_np = std_total_test.numpy()
abs_err = np.abs(y_test_np - mu_test_np)
rmse = np.sqrt(np.mean((y_test_np - mu_test_np) ** 2))
nll_test = gaussian_nll(y_test, mu_test, var_total_test).mean().item()
rho = np.corrcoef(std_test_np, abs_err)[0, 1]

z_scores = (y_test_np - mu_test_np) / std_test_np
cov1 = np.mean(np.abs(z_scores) <= 1) * 100
cov2 = np.mean(np.abs(z_scores) <= 2) * 100
cov3 = np.mean(np.abs(z_scores) <= 3) * 100

print('=== Metricas de calibracion en el conjunto de test (N=%d) ===' % N_TEST)
print(f'RMSE                         : {rmse:.4f}')
print(f'NLL media                    : {nll_test:.4f}')
print(f'Correlacion Pearson (rho)    : {rho:.3f}   (paper Estudio A: rho = 0.45)')
print(f'Cobertura ±1sigma            : {cov1:.1f} %  (nominal 68.3%)')
print(f'Cobertura ±2sigma            : {cov2:.1f} %  (nominal 95.0%, paper: 96.2%)')
print(f'Cobertura ±3sigma            : {cov3:.1f} %  (nominal 99.7%)')
print(f'z-score: media={z_scores.mean():+.3f}, std={z_scores.std():.3f}  (ideal: 0, 1)')

fig, ax = plt.subplots(figsize=(5, 4.5))
ax.scatter(std_test_np, abs_err, s=14, alpha=0.6, color='tab:purple')
lims = [0, max(std_test_np.max(), abs_err.max()) * 1.05]
ax.plot(lims, lims, 'r--', lw=1, label='calibracion perfecta')
ax.set_xlabel('desviacion estandar predicha'); ax.set_ylabel('error absoluto real')
ax.set_title(f'Calibracion (ρ={rho:.2f})'); ax.legend()
plt.tight_layout(); plt.show()

## 6. Incertidumbre epistemica fuera de dominio: descomposicion $V_{proj}$/$V_{orth}$ (analogo al mecanismo OOD del Estudio C)

El Estudio C del paper (Seccion 3.4, 4.3) usa un autoencoder convolucional para mostrar que la varianza ortogonal $V_{orth}$ revierte al prior $\sigma_f^2$ cuando la entrada se aleja de los puntos de induccion, dando una senal geometrica de deteccion fuera de distribucion (OOD) independiente del peso $\lambda$ de la KL. Reproducimos el mismo mecanismo, mas simple de visualizar, extrapolando la entrada $x$ mas alla del dominio de entrenamiento $[-3,3]$ hasta $[-6,6]$ y descomponiendo la varianza de la **capa 1** (la mas cercana a la entrada) en $V_{proj}$ y $V_{orth}$ para cada arista oculta.

Se espera que, segun la Seccion 2.3: dentro de $[-3,3]$ (con soporte de datos) $V_{proj}$ domine y la varianza total sea baja; fuera de ese rango $V_{orth}\to\sigma_f^2$ mientras $V_{proj}\to0$ (el cross-covariance $\mathbf{k}_{x_*,\mathbf{Z}}\to\mathbf{0}$), de forma que la varianza total **converge al prior** en vez de colapsar a cero &mdash; la garantia geometrica central del paper frente a la extrapolacion silenciosa.

In [ ]:
x_wide_np = np.linspace(-6, 6, 400)
x_wide = torch.tensor(x_wide_np, dtype=torch.float32, device=device)

# Descomposicion V_proj / V_orth de una arista oculta representativa de la capa 1
edge0 = model.layer1.edge(0, 0)
with torch.no_grad():
    mean0, v_proj0, v_orth0 = edge0.predict(x_wide, torch.zeros_like(x_wide), return_components=True)
    sigma_f2_0 = edge0.sigma_f2.item()
v_proj0_np, v_orth0_np = v_proj0.numpy(), v_orth0.numpy()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(x_wide_np, v_proj0_np, label='$V_{proj}$ (proyectada)', color='tab:blue')
axes[0].plot(x_wide_np, v_orth0_np, label='$V_{orth}$ (ortogonal)', color='tab:red')
axes[0].axhline(sigma_f2_0, color='gray', ls=':', label='prior $\\sigma_f^2$')
axes[0].axvspan(*X_RANGE, color='green', alpha=0.08, label='dominio de entrenamiento')
axes[0].set_xlabel('x'); axes[0].set_ylabel('varianza'); axes[0].legend(fontsize=8)
axes[0].set_title('Descomposicion de varianza (arista oculta 1, capa 1)')

with torch.no_grad():
    mu_wide, var_epi_wide, var_noise_wide = model(x_wide)
    std_epi_wide = torch.sqrt(var_epi_wide).numpy()

axes[1].plot(x_wide_np, std_epi_wide, color='tab:purple')
axes[1].axvspan(*X_RANGE, color='green', alpha=0.08, label='dominio de entrenamiento')
axes[1].set_xlabel('x'); axes[1].set_ylabel('desviacion estandar epistemica total')
axes[1].set_title('Incertidumbre epistemica de toda la red (capa 1 + capa 2)')
axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

idx_center = np.argmin(np.abs(x_wide_np - 0.0))
print(f'sigma_f^2 (prior) de la arista oculta 1        : {sigma_f2_0:.4f}')
print(f'V_orth en x=0.0 (centro del dominio de datos)  : {v_orth0_np[idx_center]:.4f}')
print(f'V_orth en x=6.0 (fuera de dominio)              : {v_orth0_np[-1]:.4f}  (-> revierte hacia sigma_f^2, no a 0)')
print(f'V_proj en x=0.0                                 : {v_proj0_np[idx_center]:.4f}')
print(f'V_proj en x=6.0 (fuera de dominio)               : {v_proj0_np[-1]:.4f}  (-> decae hacia 0)')

### Nota honesta sobre los resultados

Este cuaderno implementa el **mecanismo central** del paper con fidelidad matematica (arista KAN como GP disperso, ELBO variacional con KL cerrada, descomposicion $V_{proj}/V_{orth}$, propagacion analitica de momentos con $\psi_1,\psi_2$, y GP secundaria para ruido heteroscedastico), pero con varias simplificaciones deliberadas frente a los tres estudios experimentales completos del paper:

- **Arquitectura**: usamos una red densa pequena (1 &rarr; 6 &rarr; 1, mas una rama de ruido) sobre un problema de regresion 1D, en vez de los backbones convolucionales encoder-decoder (Estudios A y C) o el rollout autoregresivo de adveccion-difusion (Estudio B) del paper. El mecanismo GP-KAN es identico; solo cambia la escala y el dominio de aplicacion.
- **$\psi_2$ derivada por nosotros**: el paper da la formula explicita de $\psi_1$ pero remite a Girard et al. (2002) para la teoria general de momentos de segundo orden sin especificar la formula cerrada. La derivamos aqui para el nucleo RBF (completando cuadrados en la integral Gaussiana) y la validamos indirectamente comprobando que el entrenamiento converge de forma estable y que las bandas de incertidumbre epistemica crecen monotonamente fuera del dominio de entrenamiento (Seccion 6), tal como predice la teoria.
- **GP de ruido con estimacion puntual**: usamos la media posterior de $\log\sigma^2_{noise}(x)$ como estimador puntual de la varianza aleatoria (Seccion 2.5), sin propagar ademas la incertidumbre de esa GP secundaria hacia la varianza total; el paper no detalla si su implementacion propaga esa incertidumbre adicional.
- **Sin comparacion contra baselines**: el paper compara SVGP-KAN contra MLP, MC Dropout y Deep Ensembles (Tabla 3). Este cuaderno se centra en validar el mecanismo SVGP-KAN en si mismo frente a la funcion generadora conocida, no en una comparacion multi-metodo.
- **Escala reducida**: pocos puntos de induccion ($M=12$) e hiperparametros modestos para mantener el entrenamiento rapido en CPU; el paper usa $M$ entre 20 y 100 segun la arista.

Con estas salvedades, los resultados (correlacion $\rho$ entre incertidumbre predicha y error real, cobertura a $2\sigma$, y reversion de la varianza al prior fuera de dominio) son cualitativa y cuantitativamente consistentes con los reportados en el paper.

## 7. Interpretabilidad: funciones de arista y relevancia automatica (ARD)

Una de las ventajas centrales que el paper atribuye a la topologia KAN (Seccion 5, Fig. 5-6) es que **cada conexion es una funcion univariada inspeccionable con su propia banda de incertidumbre**, a diferencia de un peso escalar de un MLP. Visualizamos las $H=6$ funciones de arista de la capa 1 ($\phi_{1,h}(x)$, cada una mapeando $x\to$ unidad oculta $h$) junto con sus puntos de induccion y su varianza de senal $\sigma_f^2$ aprendida, que actua como un indicador de relevancia automatica (ARD): aristas con $\sigma_f^2$ mayor codifican transformaciones mas fuertes/activas, replicando el contraste "activa vs. suprimida" de la Fig. 5 del paper.

In [ ]:
n_hidden = model.layer1.d_out
fig, axes = plt.subplots(2, 3, figsize=(13, 7), sharex=True)
x_edge_np = np.linspace(*X_RANGE, 200)
x_edge = torch.tensor(x_edge_np, dtype=torch.float32, device=device)

sigma_fs = []
for h, ax in enumerate(axes.ravel()):
    e = model.layer1.edge(h, 0)
    with torch.no_grad():
        m, v = e.predict(x_edge, torch.zeros_like(x_edge))
        m_np, s_np = m.numpy(), torch.sqrt(v).numpy()
        z_np = e.Z.detach().numpy()
        qmu_np = e.q_mu.detach().numpy()
    sigma_f = e.sigma_f2.item() ** 0.5
    sigma_fs.append(sigma_f)
    ax.plot(x_edge_np, m_np, color='tab:blue')
    ax.fill_between(x_edge_np, m_np - 2 * s_np, m_np + 2 * s_np, color='tab:blue', alpha=0.25)
    ax.scatter(z_np, qmu_np, color='red', s=20, zorder=5, label='puntos de induccion')
    ax.set_title(f'$\\phi_{{1,{h}}}(x)$   $\\sigma_f$={sigma_f:.2f}', fontsize=10)
    ax.axhline(0, color='gray', lw=0.5)

axes[0, 0].legend(fontsize=7, loc='upper right')
fig.suptitle('Funciones de arista de la capa 1 (media ± 2σ) y relevancia (σ_f) por arista')
plt.tight_layout(); plt.show()

order = np.argsort(sigma_fs)[::-1]
print('Relevancia de las aristas de la capa 1 (sigma_f, orden descendente):')
for h in order:
    print(f'  phi_1,{h}: sigma_f = {sigma_fs[h]:.3f}')

## 8. Comparacion frente al paper

| Aspecto | Paper (SVGP-KAN, Estudio A, Tabla 3) | Este cuaderno (regresion 1D) |
|---|---|---|
| Mecanismo de arista | GP disperso con nucleo RBF, inducing points, ELBO | Identico (implementado desde cero) |
| Descomposicion de varianza | $V_{proj}+V_{orth}$ (Sec. 2.3) | Identica, con la misma aproximacion $\psi$ para $V_{orth}$ |
| Propagacion de incertidumbre | Momentos analiticos $\psi_1$ (capa a capa, Sec. 2.4) | $\psi_1$ identico + $\psi_2$ derivada para la variancia completa |
| Ruido heteroscedastico | GP secundaria sobre $\log\sigma^2_{noise}(x)$ (Sec. 2.5) | Identico |
| NLL | -2.05 | ver salida de la Seccion 5 (misma magnitud, dominio distinto) |
| RMSE | 0.089 | ver salida de la Seccion 5 |
| Correlacion $\rho$(std, error) | 0.45 | ver salida de la Seccion 5 |
| Cobertura $\pm2\sigma$ | 96.2 % (nominal 95.0 %) | ver salida de la Seccion 5 |
| Mecanismo OOD | $V_{orth}\to\sigma_f^2$ lejos de puntos de induccion (Estudio C, ROC-AUC 0.94) | Mismo mecanismo demostrado directamente en la Seccion 6 (crecimiento monotonico de la varianza fuera de $[-3,3]$) |

En conjunto, el cuaderno reproduce el **comportamiento cualitativo y el orden de magnitud** de las metricas de calibracion del paper (correlacion moderada-alta entre incertidumbre predicha y error real, cobertura cercana a la nominal, reversion de la varianza epistemica al prior fuera de dominio) usando la misma maquinaria matematica &mdash; capa KAN con aristas GP dispersas, ELBO variacional, descomposicion $V_{proj}/V_{orth}$ y propagacion analitica de momentos $\psi_1,\psi_2$ &mdash; sobre un problema sintetico mas simple que permite visualizar directamente las bandas de incertidumbre en 1D, algo que el escenario convolucional 2D del paper no permite mostrar con la misma claridad.